# Production Technology

The dataset contains `N = 441` firms observed over `T = 12` years, 1968-1979. There variables are: 
* `lcap`: Log of capital stock, $k_{it}$ 
* `lemp`: log of employment, $\ell_{it}$ 
* `ldsa`: log of deflated sales, $y_{it}$
* `year`: the calendar year of the observation, `year` $ = 1968, ..., 1979$, 
* `firmid`: anonymized indicator variable for the firm, $i = 1, ..., N$, with $N=441$. 

In [ ]:
# Import all necessary packages
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# plotting
colors = plt.rcParams['axes.prop_cycle'].by_key()['color']
plt.rcParams.update({'axes.grid':True,'grid.color':'black','grid.alpha':'0.25','grid.linestyle':'--'})
plt.rcParams.update({'font.family':'serif','mathtext.fontset':'cm'})
plt.rcParams.update({'font.size': 12})

# autoreload modules when code is run
%load_ext autoreload
%autoreload 2

#Paths to folders
# TODO: Change the paths to your own folders
figures_path = 'Figures/'


We load the data and confirm that the panel is balanced.

In [ ]:
dat = pd.read_csv('firms.csv')

# i. 441 firms, each observed 12 times
counts = dat.groupby('firmid').size()
assert counts.size == 441, f'expected 441 firms, found {counts.size}'
assert (counts == 12).all(), 'panel is not balanced'

# Descriptives

In [ ]:
dat.describe()

In [ ]:
dat[['lcap','lemp','ldsa']].hist();

In [ ]:
sns.scatterplot(x='lemp', y='ldsa', data=dat); 

# Firm-level clustering

In [ ]:
panels = [('lcap',r'log capital stock $k_{it}$','output_capital.pdf'),
          ('lemp',r'log employment $\ell_{it}$','output_labor.pdf')]

# firm colors, skipping the gray in the default cycle
firm_colors = colors[:7] + colors[8:9]

for var,xlabel,filename in panels:

    fig,ax = plt.subplots(figsize=(6,5))

    # i. all observations in the background
    ax.scatter(dat[var],dat.ldsa,s=5,color='gray',alpha=0.3,label='Other firms')

    # ii. firms 1 to 8, one color per firm
    for j,color in zip(range(1,9),firm_colors):
        sub = dat[dat.firmid == j]
        ax.scatter(sub[var],sub.ldsa,s=20,color=color,label=f'Firm {j}')

    ax.set_xlabel(xlabel)
    ax.set_ylabel('log deflated sales $y_{it}$')
    ax.legend(fontsize=10)

    fig.tight_layout()
    fig.savefig(figures_path+filename,bbox_inches='tight')
    plt.show()

The figure shows that output clusters at the firm level: each firm's observations lie in a tight cloud, and firms differ in level. This points to firm-specific effects $c_i$.

In [ ]:
dat = pd.read_csv('firms.csv')

In [ ]:
dat.sample(5)

In [ ]:
dat.year.unique()

# Converting data to numpy format 

Extract data from `pandas` to `numpy` arrays. 

In [ ]:
dat.ldsa.values.shape

In [ ]:
N = dat.firmid.unique().size
T = dat.year.unique().size
assert dat.shape[0] == N*T, f'Error: data is not a balanced panel'
print(f'Data has N={N} and T={T}')

In [ ]:
y = dat.ldsa.values.reshape((N*T,1))

ones = np.ones((N*T,1))
l = dat.lemp.values.reshape((N*T,1))
k = dat.lcap.values.reshape((N*T,1))
X = np.hstack([ones, l, k])